# IDRiD External Evaluation — Experiment 2

**Evaluation only.** This notebook runs the **finalized, frozen RACAF model** over the **IDRiD
Disease Grading Testing Set (103 images)** and nothing else. It never trains, fine-tunes, tunes a
hyperparameter or threshold, selects a checkpoint, or touches the APTOS split or caches.

## Scientific question

> How well does the finalized RACAF model generalize to the IDRiD disease-grading testing set,
> without any training or tuning on IDRiD classification labels?

This is a **different question** from Experiment 1 (the NO-RACAF ablation, which measures RACAF's
contribution). The two are not combined here.

## Independence — stated honestly

IDRiD is external at the **classification** level: no IDRiD image took part in training,
validating or tuning the classifier, and the frozen Stage 04 lesion-training images do not appear
in the grading testing set (verified byte-wise in `[E4]`).

**IDRiD is NOT independent of the full pipeline.** The frozen Stage 04 lesion segmentation model
was trained on the IDRiD *segmentation* subset, and its class-balance loss weights were chosen
from those same 54 training images. Stage 03 is a vendored LWNet checkpoint trained by its authors
on DRIVE. This notebook therefore reports an **external classification** result, never a
"completely independent full-pipeline" result.

## What is frozen

| Component | Source | Status |
|---|---|---|
| Stage 02 preprocessing | `image_preprocessing.preprocess_array(profile="DR")` | unmodified |
| Stage 03 vessel | `exported_models/VesselSegmentation/best_model.pth` | frozen, SHA-verified |
| Stage 04 lesion | `exported_models/LesionSegmentation/best_model.keras` | frozen, SHA-verified |
| Stage 05/06/07 + RACAF + CORN | `exported_models/FinalClassification/2026-09-12_02-45-05_BEST` | frozen, SHA-verified |

Every one of those is loaded through the repository's own unmodified functions. No architecture or
preprocessing is reimplemented here.

## Run order

Fresh runtime. Set `RUN_EXTERNAL_EVALUATION = True` in **[E0]**, then run **[E0] … [E13]** in
order. As committed every switch is off and the notebook does nothing.

**Reads `raw/`, never `processed/`.** Drive already holds a Stage-02-preprocessed IDRiD tree;
feeding it in would apply gamma+CLAHE twice. `[E4]` asserts the raw directory is in use.


In [ ]:
# ==== [E0] SESSION CONFIGURATION -- the ONLY cell to edit before a run ====
# Evaluation only. There is no training switch in this notebook at all.
import posixpath

RUN_EXTERNAL_EVALUATION = False      # True to evaluate the frozen model on IDRiD grading TEST
BOOTSTRAP_RESAMPLES = 2000           # QWK confidence interval
BOOTSTRAP_SEED = 20260913            # fixed so the interval is reproducible; NOT a model seed
BATCH_SIZE = 2                       # inference batching only; BatchNorm uses running statistics,
                                     # so predictions are batch-size invariant

if not isinstance(RUN_EXTERNAL_EVALUATION, bool):
    raise TypeError("RUN_EXTERNAL_EVALUATION must be True or False")
print("Session role:", "EXTERNAL EVALUATION (IDRiD grading test)" if RUN_EXTERNAL_EVALUATION
      else "inspection only -- nothing will be evaluated or written")


In [ ]:
# ==== [E1] BOOTSTRAP -- clone/pull the repository and set sys.path ====
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)


In [ ]:
# ==== [E2] SETUP -- mount Drive, install requirements, verify environment, imports ====
import setup

setup_info = setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

import config
import corn
import joint_training_dataset as jtd
import joint_training_model as jtm
import local_feature_extraction_dataset as lfed
import racaf


In [ ]:
# ==== [E3] PROVENANCE TOOLKIT + FROZEN-ARTIFACT GUARDS (read-only helpers) ====
import datetime
import hashlib
import json
import math
import platform
import subprocess

import numpy as np
import tensorflow as tf

import training.checkpointing as ckpt
from training import model_precision_policies
from training.trainer import DIAGNOSTIC_DIRTY_ATTRIBUTE

FINAL_CLASSIFICATION_EXPERIMENTS_DIR = colab_config.DRIVE.experiment_dir("FinalClassification")
FINAL_CLASSIFICATION_EXPORTED_DIR = colab_config.FINAL_CLASSIFICATION_EXPORTED_DIR

# Everything this notebook must never write to, as written and fully resolved.
FROZEN_RACAF_EXPERIMENT = posixpath.join(FINAL_CLASSIFICATION_EXPERIMENTS_DIR, "2026-09-12_02-45-05")
FROZEN_RACAF_ARCHIVE = posixpath.join(FINAL_CLASSIFICATION_EXPORTED_DIR, "2026-09-12_02-45-05_BEST")
BASELINE_0_DIR = posixpath.join(FINAL_CLASSIFICATION_EXPERIMENTS_DIR, "2026-09-11_12-18-34")
PROTECTED_PATHS = (FROZEN_RACAF_EXPERIMENT, FROZEN_RACAF_ARCHIVE, BASELINE_0_DIR,
                   FINAL_CLASSIFICATION_EXPERIMENTS_DIR)

EXPECTED_MODEL_SHA256 = "7b780c230e4ce83ae9e9b1c0bdb9ae73c707b262e5a9c759748343f9417b8b9a"
EXPECTED_CONFIG_HASH = "3f549e1638d9409f7862f1e799bd7052"
EXPECTED_TRAINABLE_PARAMETERS = 43_338_506
EXPECTED_TRAINABLE_TENSORS = 409
EXPECTED_PRECISION_POLICY = "mixed_float16"
LEARNING_RATE = 1e-4          # only to reconstruct the compiled graph; the optimizer never steps
MIXED_PRECISION = True
IMAGE_SIZE = jtd.STAGE5_IMAGE_SIZE            # (512, 512)
GRADES = list(range(corn.NUM_GRADES))         # [0, 1, 2, 3, 4]


def git_output(*args):
    return subprocess.run(["git", "-C", colab_config.REPO_DIR, *args],
                          capture_output=True, text=True, check=True).stdout.strip()


def read_json(path):
    with open(path) as handle:
        return json.load(handle)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def is_protected(path):
    """True for anything at or beneath a frozen artifact -- this notebook never writes there."""
    written = posixpath.normpath(str(path))
    resolved = os.path.realpath(str(path))
    for root in PROTECTED_PATHS:
        if written == root or written.startswith(root + "/"):
            return True
        real = os.path.realpath(root)
        if resolved == real or resolved.startswith(real + os.sep):
            return True
    return False


def assert_writable(path, what):
    if is_protected(path):
        raise RuntimeError(f"Refusing to write {what} into the frozen area: {path}. "
                           "The finalized RACAF experiment, its archive and baseline-0 are immutable.")
    return path


SESSION_COMMIT = git_output("rev-parse", "HEAD")
_origin = git_output("rev-parse", "origin/" + colab_config.REPO_BRANCH)
_modified = git_output("status", "--porcelain", "--untracked-files=no")
print("repository:", colab_config.REPO_DIR)
print("  commit:", SESSION_COMMIT)
if SESSION_COMMIT != _origin:
    raise RuntimeError(f"The runtime's clone is at {SESSION_COMMIT}, not origin/"
                       f"{colab_config.REPO_BRANCH} ({_origin}). Re-run [E1].")
if _modified:
    raise RuntimeError("Tracked files were modified inside this runtime's clone:\n" + _modified)
print(f"  clean, identical to origin/{colab_config.REPO_BRANCH}")

ENVIRONMENT = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "git_commit": SESSION_COMMIT,
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": tf.keras.__version__,
    "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],
    "gpu_details": [tf.config.experimental.get_device_details(d)
                    for d in tf.config.list_physical_devices("GPU")],
    "platform": platform.platform(),
}
print(json.dumps(ENVIRONMENT, indent=2, default=str))


In [ ]:
# ==== [E4] IDRiD DATA DISCOVERY + SANITY CHECKS (read-only; raw images only) ====
import collections
import csv as _csv

IDRID_ROOT = posixpath.join(colab_config.DRIVE.datasets_root, "IDRiD", "grading", "raw")
IDRID_TEST_IMAGE_DIR = posixpath.join(IDRID_ROOT, "1. Original Images", "b. Testing Set")
IDRID_TRAIN_IMAGE_DIR = posixpath.join(IDRID_ROOT, "1. Original Images", "a. Training Set")
IDRID_TEST_LABEL_CSV = posixpath.join(
    IDRID_ROOT, "2. Groundtruths", "b. IDRiD_Disease Grading_Testing Labels.csv")
IDRID_SEGMENTATION_DIR = posixpath.join(
    colab_config.DRIVE.datasets_root, "IDRiD", "segmentation", "raw", "1. Original Images")

EXPECTED_TEST_IMAGES = 103
EXPECTED_GRADE_DISTRIBUTION = {0: 34, 1: 5, 2: 32, 3: 19, 4: 13}   # verified on Drive beforehand
LABEL_NAME_COLUMN = "Image name"
LABEL_GRADE_COLUMN = "Retinopathy grade"        # "Risk of macular edema" is deliberately UNUSED

EVAL_CHECKS = {}


def eval_check(name, ok, detail=""):
    EVAL_CHECKS[name] = (bool(ok), str(detail))
    print(("PASS  " if ok else "FAIL  ") + name + (f"  -- {detail}" if detail else ""))
    return bool(ok)


def require(name, ok, detail=""):
    if not eval_check(name, ok, detail):
        raise RuntimeError(f"SANITY CHECK FAILED: {name} ({detail}). Nothing was evaluated.")


if RUN_EXTERNAL_EVALUATION:
    require("IDRiD raw testing image directory exists", os.path.isdir(IDRID_TEST_IMAGE_DIR),
            IDRID_TEST_IMAGE_DIR)
    require("the RAW tree is in use, not processed/", "/raw/" in IDRID_TEST_IMAGE_DIR + "/"
            and "processed" not in IDRID_TEST_IMAGE_DIR,
            "processed/ already holds Stage 02 output; using it would apply gamma+CLAHE twice")
    require("IDRiD testing label CSV exists", os.path.isfile(IDRID_TEST_LABEL_CSV),
            IDRID_TEST_LABEL_CSV)

    _image_files = sorted(f for f in os.listdir(IDRID_TEST_IMAGE_DIR)
                          if os.path.splitext(f)[1].lower() in (".jpg", ".jpeg"))
    require(f"exactly {EXPECTED_TEST_IMAGES} raw testing images",
            len(_image_files) == EXPECTED_TEST_IMAGES, f"{len(_image_files)} found")

    with open(IDRID_TEST_LABEL_CSV, newline="", encoding="utf-8-sig") as handle:
        _rows = [{(k or "").strip(): (v.strip() if isinstance(v, str) else v)
                  for k, v in row.items() if k is not None and (k or "").strip()}
                 for row in _csv.DictReader(handle)]
    _rows = [r for r in _rows if r.get(LABEL_NAME_COLUMN)]
    require(f"exactly {EXPECTED_TEST_IMAGES} label rows", len(_rows) == EXPECTED_TEST_IMAGES,
            f"{len(_rows)} rows")
    require("label CSV exposes the two columns this evaluation uses",
            all(c in _rows[0] for c in (LABEL_NAME_COLUMN, LABEL_GRADE_COLUMN)), list(_rows[0]))

    # id_code -> grade. IDRiD numbers train and test both from 001, so the evaluation id is
    # namespaced: IDRiDtest_<nnn>. This is what keeps the cache free of collisions.
    LABELS, _bad_grades = {}, []
    for row in _rows:
        stem = os.path.splitext(str(row[LABEL_NAME_COLUMN]).strip())[0]
        try:
            grade = int(float(row[LABEL_GRADE_COLUMN]))
        except (TypeError, ValueError):
            _bad_grades.append((stem, row.get(LABEL_GRADE_COLUMN))); continue
        if grade not in GRADES:
            _bad_grades.append((stem, grade)); continue
        LABELS[stem] = grade
    require("every grade parses and lies in {0,1,2,3,4}", not _bad_grades, _bad_grades[:8])
    require("no duplicate label rows", len(LABELS) == len(_rows), f"{len(LABELS)} unique")

    _image_stems = [os.path.splitext(f)[0] for f in _image_files]
    require("no duplicate image basenames", len(set(_image_stems)) == len(_image_stems))
    require("image ids and label ids match one-to-one", set(_image_stems) == set(LABELS),
            f"image-only={sorted(set(_image_stems) - set(LABELS))[:5]} "
            f"label-only={sorted(set(LABELS) - set(_image_stems))[:5]}")

    _observed = collections.Counter(LABELS.values())
    require("grade distribution matches the pre-verified Drive distribution",
            {g: _observed.get(g, 0) for g in GRADES} == EXPECTED_GRADE_DISTRIBUTION,
            f"{dict(sorted(_observed.items()))} vs {EXPECTED_GRADE_DISTRIBUTION}")

    EVALUATION_ENTRIES = [("IDRiDtest_" + stem.split("_")[-1], stem, grade)
                          for stem, grade in sorted(LABELS.items())]
    require("evaluation ids are unique and namespaced",
            len({e[0] for e in EVALUATION_ENTRIES}) == EXPECTED_TEST_IMAGES
            and all(e[0].startswith("IDRiDtest_") for e in EVALUATION_ENTRIES))
    require("no training image is referenced",
            all(os.path.dirname(posixpath.join(IDRID_TEST_IMAGE_DIR, s + ".jpg"))
                == IDRID_TEST_IMAGE_DIR for _, s, _ in EVALUATION_ENTRIES))

    # ---- data-integrity: byte-identical overlap, size-prefiltered ------------------------------
    def _by_size(directory, exts=(".jpg", ".jpeg")):
        out = collections.defaultdict(list)
        if not os.path.isdir(directory):
            return out
        for entry in os.scandir(directory):
            if entry.is_file() and os.path.splitext(entry.name)[1].lower() in exts:
                out[entry.stat().st_size].append(entry.path)
        return out

    def _byte_overlap(dir_a, dir_b):
        a, b = _by_size(dir_a), _by_size(dir_b)
        pairs = []
        for size in set(a) & set(b):
            hashes = {}
            for p in a[size]:
                hashes.setdefault(sha256_file(p), []).append(p)
            for q in b[size]:
                h = sha256_file(q)
                for p in hashes.get(h, []):
                    pairs.append((os.path.basename(p), os.path.basename(q), h))
        return pairs

    TEST_VS_IDRID_TRAIN = _byte_overlap(IDRID_TRAIN_IMAGE_DIR, IDRID_TEST_IMAGE_DIR)
    print(f"\nbyte-identical IDRiD TRAIN vs TEST images: {len(TEST_VS_IDRID_TRAIN)}")
    for a, b, h in TEST_VS_IDRID_TRAIN:
        print(f"   TRAIN {a} == TEST {b}  sha256={h[:16]}...")
    # IDRiD's own train split is never used by this pipeline, so an internal IDRiD duplicate
    # leaks nothing into this evaluation. It is recorded, not silently ignored, and not excluded.
    eval_check("IDRiD train/test duplicates recorded (not leakage for this model)", True,
               f"{len(TEST_VS_IDRID_TRAIN)} pair(s); IDRiD train is not used anywhere in this pipeline")

    _seg_dirs = [posixpath.join(IDRID_SEGMENTATION_DIR, d)
                 for d in ("a. Training Set", "b. Testing Set")
                 if os.path.isdir(posixpath.join(IDRID_SEGMENTATION_DIR, d))]
    SEG_VS_TEST = []
    for d in _seg_dirs:
        SEG_VS_TEST += _byte_overlap(d, IDRID_TEST_IMAGE_DIR)
    print(f"\nbyte-identical Stage-04 lesion-training vs IDRiD TEST images: {len(SEG_VS_TEST)}")
    for a, b, h in SEG_VS_TEST:
        print(f"   SEG {a} == TEST {b}  sha256={h[:16]}...")
    # THIS one would be real leakage: those images trained the frozen Stage 04.
    require("no Stage-04 lesion-training image appears in the IDRiD TEST set",
            not SEG_VS_TEST, f"{len(SEG_VS_TEST)} overlapping image(s) -- evaluation would be contaminated")

    print(f"\n{len(EVALUATION_ENTRIES)} IDRiD testing images will be evaluated.")
    print("true grade distribution:", dict(sorted(_observed.items())))
else:
    EVALUATION_ENTRIES, LABELS, TEST_VS_IDRID_TRAIN, SEG_VS_TEST = [], {}, [], []
    print("RUN_EXTERNAL_EVALUATION is False -- no data was read.")


In [ ]:
# ==== [E5] FROZEN CHECKPOINT VERIFICATION (hashes only; nothing is loaded yet) ====
VESSEL_CHECKPOINT = posixpath.join(config.VESSEL_SEG_MODEL_DIR, "best_model.pth")
LESION_CHECKPOINT = posixpath.join(config.LESION_SEG_MODEL_DIR, "best_model.keras")
ARCHIVED_WEIGHTS = posixpath.join(FROZEN_RACAF_ARCHIVE, ckpt.MODEL_WEIGHTS_FILENAME)

CHECKPOINT_PROVENANCE = {}
if RUN_EXTERNAL_EVALUATION:
    require("frozen RACAF archive exists", os.path.isdir(FROZEN_RACAF_ARCHIVE), FROZEN_RACAF_ARCHIVE)
    require("archived model.weights.h5 exists", os.path.isfile(ARCHIVED_WEIGHTS), ARCHIVED_WEIGHTS)
    require("vessel checkpoint exists", os.path.isfile(VESSEL_CHECKPOINT), VESSEL_CHECKPOINT)
    require("lesion checkpoint exists", os.path.isfile(LESION_CHECKPOINT), LESION_CHECKPOINT)

    _model_sha = sha256_file(ARCHIVED_WEIGHTS)
    require(f"archived model SHA-256 is the expected frozen value",
            _model_sha == EXPECTED_MODEL_SHA256, f"{_model_sha} vs {EXPECTED_MODEL_SHA256}")

    # "ARCHIVE_READY" is the marker [D11] of the frozen notebook wrote; it is a notebook-level
    # convention, not a constant exported by training.checkpointing, so it is spelled out here.
    ARCHIVE_READY_FILENAME = "ARCHIVE_READY"
    _provenance = read_json(posixpath.join(FROZEN_RACAF_ARCHIVE, "provenance.json"))
    _ready_path = posixpath.join(FROZEN_RACAF_ARCHIVE, ARCHIVE_READY_FILENAME)
    require("the archive carries its READY marker", os.path.isfile(_ready_path), _ready_path)
    _ready = read_json(_ready_path)
    require("archive is sealed and its READY marker agrees with the weights",
            _ready.get("model_sha256") == _model_sha)
    require("archive provenance is the frozen experiment and its config hash",
            _provenance.get("experiment_id") == "2026-09-12_02-45-05"
            and _provenance.get("config_hash") == EXPECTED_CONFIG_HASH,
            f"{_provenance.get('experiment_id')} / {_provenance.get('config_hash')}")

    CHECKPOINT_PROVENANCE = {
        "racaf_archive": FROZEN_RACAF_ARCHIVE,
        "racaf_model_weights": ARCHIVED_WEIGHTS,
        "racaf_model_sha256": _model_sha,
        "racaf_config_hash": _provenance.get("config_hash"),
        "racaf_experiment_id": _provenance.get("experiment_id"),
        "racaf_best_epoch": _provenance.get("best_epoch"),
        "racaf_best_val_qwk_recorded": _provenance.get("best_val_qwk_recorded"),
        "racaf_git_commit": _provenance.get("git_commit"),
        "vessel_checkpoint": VESSEL_CHECKPOINT,
        "vessel_sha256": sha256_file(VESSEL_CHECKPOINT),
        "lesion_checkpoint": LESION_CHECKPOINT,
        "lesion_sha256": sha256_file(LESION_CHECKPOINT),
    }
    print(json.dumps(CHECKPOINT_PROVENANCE, indent=2, default=str))
else:
    print("RUN_EXTERNAL_EVALUATION is False -- no checkpoint was verified.")


In [ ]:
# ==== [E6] RECONSTRUCT THE FROZEN MODEL AND LOAD THE ARCHIVED WEIGHTS ====
# Exactly the reconstruction the frozen notebook's own final integrity check performs: build the
# production architecture via the unmodified builder, then load ONLY the archived weights. The
# optimizer exists solely because compile() requires one; it is never stepped, and the model is
# marked diagnostic-dirty so training.Trainer.fit() would refuse it outright.
if RUN_EXTERNAL_EVALUATION:
    joint_model = jtm.build_and_compile_joint_model(
        mixed_precision=MIXED_PRECISION,
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), verbose=0)
    setattr(joint_model, DIAGNOSTIC_DIRTY_ATTRIBUTE, True)
    ckpt.load_model_weights_only(joint_model, ARCHIVED_WEIGHTS)

    _parameters = sum(int(np.prod(v.shape)) for v in joint_model.trainable_variables)
    _layer_names = {layer.name for layer in joint_model.layers}
    _variable_paths = " ".join(v.path for v in joint_model.trainable_variables).lower()
    require("model is the production joint_stage05_08_racaf graph",
            joint_model.name == "joint_stage05_08_racaf", joint_model.name)
    require(f"{EXPECTED_TRAINABLE_PARAMETERS:,} trainable parameters",
            _parameters == EXPECTED_TRAINABLE_PARAMETERS, f"{_parameters:,}")
    require(f"{EXPECTED_TRAINABLE_TENSORS} trainable tensors",
            len(joint_model.trainable_variables) == EXPECTED_TRAINABLE_TENSORS,
            len(joint_model.trainable_variables))
    require("RACAF IS ACTIVE in the graph", "racaf_fusion" in _layer_names
            and "racaf" in _variable_paths, sorted(_layer_names))
    require("model output is (None, 4) CORN logits",
            tuple(joint_model.outputs[0].shape) == (None, corn.NUM_THRESHOLDS),
            str(tuple(joint_model.outputs[0].shape)))
    require("layers are under the frozen precision policy",
            EXPECTED_PRECISION_POLICY in model_precision_policies(joint_model),
            sorted(model_precision_policies(joint_model)))
    require("the optimizer has never stepped",
            int(tf.keras.backend.get_value(joint_model.optimizer.iterations)) == 0)
    require("model is marked diagnostic-dirty (Trainer.fit would refuse it)",
            getattr(joint_model, DIAGNOSTIC_DIRTY_ATTRIBUTE, False) is True)

    print(f"\nfrozen model reconstructed: {_parameters:,} trainable parameters, "
          f"{len(joint_model.trainable_variables)} tensors, output "
          f"{tuple(joint_model.outputs[0].shape)}")
    print("Stage 03/04 are loaded separately in [E7]; Stage 01 IQA is not part of this graph.")
else:
    joint_model = None
    print("RUN_EXTERNAL_EVALUATION is False -- no model was built.")


In [ ]:
# ==== [E7] FROZEN PIPELINE PER IMAGE -> namespaced IDRiD cache (Stage 02/03/04 + RACAF r) ====
# Every stage below is the repository's own unmodified function. Nothing is reimplemented:
#   cv2.imread            -> raw BGR (JPG; the APTOS loader is .png-only, which is why the raw
#                            read happens here rather than through lfed._load_raw_bgr)
#   lfed._stage02_processed_rgb  -> Stage 02 gamma 1.2 + CLAHE 2.0/(8,8), then BGR->RGB
#   jtd._get_or_compute_joint_frozen_outputs -> Stage 03 vessel, Stage 04 lesion, RACAF kappa/r
#       from ONE deterministic racaf.tta_views() call (identity, h-flip, v-flip, rot180)
#   jtd._get_or_compute_canonical_rgb        -> channels 0-2 of stage5_input
#   lfed._resize_input                       -> stage6_input from stage5's own RGB
# The 8-channel assembly mirrors jtd._build_joint_sample exactly, minus augmentation.
import shutil

import cv2

IDRID_CACHE_DIR = "/content/idrid_cache/local_feature_extraction"
IDRID_RACAF_CACHE_DIR = "/content/idrid_cache/racaf"
EXCLUSIONS = []
PIPELINE_STATS = {"computed": 0, "cache_hits": 0}

if RUN_EXTERNAL_EVALUATION:
    for d in (IDRID_CACHE_DIR, IDRID_RACAF_CACHE_DIR):
        assert_writable(d, "the IDRiD cache")
        os.makedirs(d, exist_ok=True)
    # The cache lives in its OWN directory under /content and is keyed by IDRiDtest_* ids, so it
    # can never collide with, or be mistaken for, the APTOS cache. The filenames still carry the
    # repository's historical "APTOS_" prefix because lfed._cache_path builds them and this
    # notebook does not modify that function -- the directory and the id are what namespace it.
    require("IDRiD cache directory is not the APTOS cache",
            os.path.realpath(IDRID_CACHE_DIR) != os.path.realpath(config.LOCAL_FEATURE_RESULTS_DIR)
            and os.path.realpath(IDRID_RACAF_CACHE_DIR) != os.path.realpath(config.RACAF_RESULTS_DIR))

    vessel_model = jtd.load_vessel_model(jtd.DEFAULT_VESSEL_MODEL_PATH)
    stage4_model = racaf.load_frozen_stage4_model()
    print("frozen Stage 03 and Stage 04 loaded (inference only).\n")

    def build_sample(evaluation_id, stem):
        """One evaluation sample through the frozen pipeline. Returns the sample dict, or raises."""
        image_path = posixpath.join(IDRID_TEST_IMAGE_DIR, stem + ".jpg")
        vessel_cache = lfed._cache_path(IDRID_CACHE_DIR, evaluation_id, "vessel", IMAGE_SIZE)
        lesion_cache = lfed._cache_path(IDRID_CACHE_DIR, evaluation_id, "lesion", IMAGE_SIZE)
        reliability_cache = racaf.reliability_cache_path(IDRID_RACAF_CACHE_DIR, evaluation_id)
        rgb_cache = jtd._canonical_rgb_cache_path(evaluation_id, IDRID_CACHE_DIR, IMAGE_SIZE)
        cached = all(os.path.exists(p) for p in (vessel_cache, lesion_cache, reliability_cache, rgb_cache))

        rgb_native = None
        if not cached:
            raw_bgr = cv2.imread(image_path)
            if raw_bgr is None:
                raise ValueError(f"cv2.imread returned None for {image_path}")
            rgb_native = lfed._stage02_processed_rgb(raw_bgr)

        vessel_map, lesion_maps, kappa, r = jtd._get_or_compute_joint_frozen_outputs(
            rgb_native, vessel_cache, lesion_cache, reliability_cache,
            vessel_model, stage4_model, image_size=IMAGE_SIZE, id_code=evaluation_id)
        canonical_rgb = jtd._get_or_compute_canonical_rgb(
            rgb_native, rgb_cache, IMAGE_SIZE, id_code=evaluation_id)

        stage5_input = np.concatenate([canonical_rgb, vessel_map, lesion_maps], axis=-1)
        stage6_input = lfed._resize_input(stage5_input[..., :3], jtd.STAGE6_IMAGE_SIZE)
        PIPELINE_STATS["cache_hits" if cached else "computed"] += 1
        return {"image_id": evaluation_id, "source_image": image_path,
                "stage5_input": stage5_input.astype(np.float32),
                "stage6_input": stage6_input.astype(np.float32),
                "reliability": float(r), "kappa": np.asarray(kappa, dtype=np.float32)}

    SAMPLES = []
    for index, (evaluation_id, stem, grade) in enumerate(EVALUATION_ENTRIES, start=1):
        try:
            sample = build_sample(evaluation_id, stem)
        except Exception as error:                      # noqa: BLE001 -- reported, never silent
            reason = f"{type(error).__name__}: {error}"
            EXCLUSIONS.append({"image_id": evaluation_id, "source_stem": stem,
                               "true_grade": grade, "reason": reason})
            print(f"  [{index:3d}/{len(EVALUATION_ENTRIES)}] {evaluation_id}: EXCLUDED -- {reason}")
            continue
        sample["true_grade"] = grade
        sample["source_stem"] = stem
        SAMPLES.append(sample)
        if index % 10 == 0 or index == len(EVALUATION_ENTRIES):
            print(f"  [{index:3d}/{len(EVALUATION_ENTRIES)}] prepared "
                  f"(computed {PIPELINE_STATS['computed']}, cache hits {PIPELINE_STATS['cache_hits']})")

    _r = np.array([s["reliability"] for s in SAMPLES], dtype=np.float64)
    RELIABILITY_SUMMARY = {
        "count": int(_r.size),
        "min": float(_r.min()) if _r.size else None, "max": float(_r.max()) if _r.size else None,
        "mean": float(_r.mean()) if _r.size else None, "std": float(_r.std()) if _r.size else None,
        "non_finite": int((~np.isfinite(_r)).sum()),
        "outside_unit_interval": int(((_r < 0.0) | (_r > 1.0)).sum()),
        "tta_views": list(racaf.TTA_TRANSFORMS),
    }
    print(f"\nprepared {len(SAMPLES)} sample(s); excluded {len(EXCLUSIONS)}")
    print("RACAF reliability:", json.dumps(RELIABILITY_SUMMARY, indent=2))
    eval_check("every reliability value is finite", RELIABILITY_SUMMARY["non_finite"] == 0)
    eval_check("every reliability value lies in [0, 1]",
               RELIABILITY_SUMMARY["outside_unit_interval"] == 0)
    eval_check("RACAF used the four deterministic TTA views",
               tuple(racaf.TTA_TRANSFORMS) ==
               ("identity", "horizontal_flip", "vertical_flip", "rotate_180"),
               str(racaf.TTA_TRANSFORMS))
    require("at least one image survived the frozen pipeline", bool(SAMPLES))
else:
    SAMPLES, RELIABILITY_SUMMARY = [], {}
    print("RUN_EXTERNAL_EVALUATION is False -- no image was processed.")


In [ ]:
# ==== [E8] INFERENCE + CORN DECODE (no optimizer, no gradients, no writes) ====
# predict_on_batch only. Deterministic by construction: a fixed, sorted image order, no shuffling
# and no augmentation anywhere in this notebook. BatchNorm runs in inference mode (running
# statistics), so the predictions do not depend on BATCH_SIZE.
if RUN_EXTERNAL_EVALUATION:
    _logit_batches = []
    for start in range(0, len(SAMPLES), BATCH_SIZE):
        chunk = SAMPLES[start:start + BATCH_SIZE]
        stage5 = np.stack([s["stage5_input"] for s in chunk])
        stage6 = np.stack([s["stage6_input"] for s in chunk])
        reliability = np.array([[s["reliability"]] for s in chunk], dtype=np.float32)
        _logit_batches.append(np.asarray(
            joint_model.predict_on_batch([stage5, stage6, reliability]), dtype=np.float32))
    LOGITS = np.concatenate(_logit_batches)
    Y_TRUE = np.array([s["true_grade"] for s in SAMPLES], dtype=int)
    EVALUATED_IDS = [s["image_id"] for s in SAMPLES]

    # The production decoder, unmodified: p_cond = sigmoid(z); p_cum = cumprod(p_cond);
    # grade = count(p_cum > 0.5). NOT a 5-class softmax.
    DECODED = corn.decode_logits(LOGITS)
    Y_PRED = DECODED["predicted_grade"].astype(int)
    P_CUM = DECODED["p_cum"].astype(np.float64)
    CLASS_PROBABILITIES = DECODED["class_probabilities"].astype(np.float64)

    require("one logit row per evaluated image", LOGITS.shape == (len(SAMPLES), corn.NUM_THRESHOLDS),
            str(LOGITS.shape))
    require("all logits are finite", bool(np.isfinite(LOGITS).all()))
    eval_check("P(y>k) is non-increasing in k for every image",
               bool(np.all(np.diff(P_CUM, axis=-1) <= 1e-12)))
    eval_check("the reconstructed 5-class distribution is non-negative and sums to 1",
               bool((CLASS_PROBABILITIES >= -1e-9).all()
                    and np.allclose(CLASS_PROBABILITIES.sum(axis=-1), 1.0, atol=1e-6)),
               f"max |sum-1| = {np.abs(CLASS_PROBABILITIES.sum(axis=-1) - 1.0).max():.2e}")
    eval_check("the decoded grade is reproducible from P(y>k)",
               np.array_equal(np.sum(P_CUM > 0.5, axis=-1).astype(int), Y_PRED))

    print(f"\nevaluated {len(Y_PRED)} images")
    print("true grades     :", dict(sorted(collections.Counter(Y_TRUE.tolist()).items())))
    print("predicted grades:", dict(sorted(collections.Counter(Y_PRED.tolist()).items())))
else:
    LOGITS = Y_TRUE = Y_PRED = P_CUM = CLASS_PROBABILITIES = None
    EVALUATED_IDS = []
    print("RUN_EXTERNAL_EVALUATION is False -- no inference was run.")


In [ ]:
# ==== [E9] METRICS -- QWK primary, everything else secondary ====
from evaluation import metrics as evm

QWK_WEIGHTS = np.subtract.outer(GRADES, GRADES) ** 2 / float((corn.NUM_GRADES - 1) ** 2)


def qwk_from_confusion(confusion):
    """training.metrics.QuadraticWeightedKappa.result(), in NumPy -- the project's own formula."""
    confusion = np.asarray(confusion, dtype=np.float64)
    total = confusion.sum()
    if total == 0:
        return 0.0
    expected = np.outer(confusion.sum(1), confusion.sum(0)) / total
    return float(1.0 - (QWK_WEIGHTS * confusion).sum() / max((QWK_WEIGHTS * expected).sum(), 1e-7))


if RUN_EXTERNAL_EVALUATION:
    CONFUSION = evm.confusion_matrix(Y_TRUE, Y_PRED, num_classes=corn.NUM_GRADES).astype(int)
    PER_CLASS = {}
    for grade in GRADES:
        tp = int(CONFUSION[grade, grade])
        fn = int(CONFUSION[grade, :].sum() - tp)
        fp = int(CONFUSION[:, grade].sum() - tp)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        PER_CLASS[grade] = {"support": tp + fn, "predicted": int(CONFUSION[:, grade].sum()),
                            "TP": tp, "FP": fp, "FN": fn,
                            "precision": precision, "recall": recall, "f1": f1}

    _distance = np.abs(Y_TRUE - Y_PRED)
    METRICS = {
        "evaluated": int(len(Y_TRUE)),
        "excluded": len(EXCLUSIONS),
        "qwk": qwk_from_confusion(CONFUSION),
        "qwk_sklearn": float(evm.quadratic_weighted_kappa(Y_TRUE, Y_PRED)),
        "accuracy": float(evm.accuracy(Y_TRUE, Y_PRED)),
        "balanced_accuracy": float(evm.recall(Y_TRUE, Y_PRED, average="macro")),
        "macro_f1": float(evm.f1_score(Y_TRUE, Y_PRED, average="macro")),
        "weighted_f1": float(evm.f1_score(Y_TRUE, Y_PRED, average="weighted")),
        "macro_precision": float(evm.precision(Y_TRUE, Y_PRED, average="macro")),
        "macro_recall": float(evm.recall(Y_TRUE, Y_PRED, average="macro")),
        "ordinal_mae": float(np.mean(np.abs(Y_TRUE.astype(float) - Y_PRED.astype(float)))),
        "adjacent_errors": int((_distance == 1).sum()),
        "non_adjacent_errors": int((_distance >= 2).sum()),
        "max_error_distance": int(_distance.max()),
        "true_histogram": np.bincount(Y_TRUE, minlength=corn.NUM_GRADES).tolist(),
        "prediction_histogram": np.bincount(Y_PRED, minlength=corn.NUM_GRADES).tolist(),
        "per_class": PER_CLASS,
        "confusion_matrix": CONFUSION.tolist(),
    }
    eval_check("the project's QWK and sklearn's quadratic kappa agree (|diff| < 1e-6)",
               abs(METRICS["qwk"] - METRICS["qwk_sklearn"]) < 1e-6)
    eval_check("the confusion matrix accounts for every evaluated image",
               int(CONFUSION.sum()) == len(Y_TRUE))

    print("=== IDRiD EXTERNAL EVALUATION -- frozen RACAF model ===")
    for key in ("evaluated", "excluded", "qwk", "accuracy", "balanced_accuracy", "macro_f1",
                "weighted_f1", "ordinal_mae"):
        value = METRICS[key]
        print(f"  {key:20} {value:.6f}" if isinstance(value, float) else f"  {key:20} {value}")
    print(f"\n{'grade':>6}{'support':>9}{'pred':>7}{'precision':>12}{'recall':>10}{'F1':>10}")
    for grade in GRADES:
        row = PER_CLASS[grade]
        print(f"{grade:>6}{row['support']:>9}{row['predicted']:>7}"
              f"{row['precision']:>12.4f}{row['recall']:>10.4f}{row['f1']:>10.4f}")
    print("\nconfusion matrix (rows = true grade, columns = predicted grade):")
    print("        " + "".join(f"{'pred ' + str(g):>9}" for g in GRADES))
    for grade in GRADES:
        print(f"true {grade} " + "".join(f"{v:>9d}" for v in CONFUSION[grade]))
else:
    METRICS, CONFUSION, PER_CLASS = {}, None, {}
    print("RUN_EXTERNAL_EVALUATION is False -- no metrics were computed.")


In [ ]:
# ==== [E10] CALIBRATION -- Brier and ECE from the CORN reconstruction ====
# The model emits FOUR cumulative-threshold logits, not five softmax logits. The five-class
# distribution used here is the finite-difference reconstruction of P(y>k), which [E8] verified is
# non-negative and sums to 1: P(y=0)=1-p0, P(y=k)=p_{k-1}-p_k, P(y=4)=p3. That is what makes a
# Brier score and an ECE well defined; nothing is treated as a softmax over four logits.
if RUN_EXTERNAL_EVALUATION:
    CALIBRATION_BINS = 10
    _one_hot = np.eye(corn.NUM_GRADES)[Y_TRUE]
    BRIER = float(np.mean(np.sum((CLASS_PROBABILITIES - _one_hot) ** 2, axis=1)))
    ECE_ARGMAX = float(evm.expected_calibration_error(
        Y_TRUE, CLASS_PROBABILITIES, n_bins=CALIBRATION_BINS))

    _confidence = CLASS_PROBABILITIES[np.arange(len(Y_PRED)), Y_PRED]
    _correct = (Y_TRUE == Y_PRED)
    _edges = np.linspace(0.0, 1.0, CALIBRATION_BINS + 1)
    _bins = np.clip(np.digitize(_confidence, _edges[1:-1], right=True), 0, CALIBRATION_BINS - 1)
    RELIABILITY_BINS, _ece_decode = [], 0.0
    for b in range(CALIBRATION_BINS):
        mask = _bins == b
        row = {"bin": b, "lower": float(_edges[b]), "upper": float(_edges[b + 1]),
               "count": int(mask.sum()),
               "mean_confidence": float(_confidence[mask].mean()) if mask.any() else None,
               "accuracy": float(_correct[mask].mean()) if mask.any() else None}
        RELIABILITY_BINS.append(row)
        if mask.any():
            _ece_decode += mask.mean() * abs(row["mean_confidence"] - row["accuracy"])
    ECE_DECODE = float(_ece_decode)

    THRESHOLD_CALIBRATION = [
        {"threshold": k, "mean_predicted": float(P_CUM[:, k].mean()),
         "empirical_rate": float((Y_TRUE > k).mean()),
         "gap": float(P_CUM[:, k].mean() - (Y_TRUE > k).mean()),
         "positives": int((Y_TRUE > k).sum())}
        for k in range(corn.NUM_THRESHOLDS)
    ]
    CALIBRATION = {
        "bins": CALIBRATION_BINS, "brier_score": BRIER,
        "ece_decode_convention": ECE_DECODE, "ece_argmax_convention": ECE_ARGMAX,
        "probability_definition": ("finite differences of the CORN cumulative probabilities "
                                   "P(y>k); verified non-negative and summing to 1"),
        "threshold_calibration": THRESHOLD_CALIBRATION,
        "reliability_bins": RELIABILITY_BINS,
        "confidence": {"mean": float(_confidence.mean()), "median": float(np.median(_confidence)),
                       "correct_mean": float(_confidence[_correct].mean()) if _correct.any() else None,
                       "incorrect_mean": float(_confidence[~_correct].mean()) if (~_correct).any() else None},
    }
    print(f"Brier score (multiclass, 0 = perfect) : {BRIER:.6f}")
    print(f"ECE, CORN decode convention           : {ECE_DECODE:.6f}")
    print(f"ECE, standard argmax convention       : {ECE_ARGMAX:.6f}")
    print(f"\n{'threshold':>10}{'mean P(y>k)':>14}{'empirical':>12}{'gap':>10}{'n(true>k)':>12}")
    for row in THRESHOLD_CALIBRATION:
        print(f"{row['threshold']:>10}{row['mean_predicted']:>14.4f}{row['empirical_rate']:>12.4f}"
              f"{row['gap']:>10.4f}{row['positives']:>12}")
    eval_check("Brier and both ECE values are finite and in range",
               all(math.isfinite(v) for v in (BRIER, ECE_DECODE, ECE_ARGMAX))
               and 0.0 <= ECE_DECODE <= 1.0 and 0.0 <= ECE_ARGMAX <= 1.0 and 0.0 <= BRIER <= 2.0)
else:
    CALIBRATION = {}
    print("RUN_EXTERNAL_EVALUATION is False -- no calibration was computed.")


In [ ]:
# ==== [E11] BOOTSTRAP 95% CONFIDENCE INTERVAL FOR QWK ====
# Percentile bootstrap over the evaluated images, resampled with replacement, QWK recomputed from
# each resample's confusion matrix with the project's own formula. The seed is fixed so the
# interval is reproducible. No IDRiD label is used to tune anything before the point estimate.
if RUN_EXTERNAL_EVALUATION:
    _rng = np.random.default_rng(BOOTSTRAP_SEED)
    _n = len(Y_TRUE)
    _samples = np.empty(BOOTSTRAP_RESAMPLES, dtype=np.float64)
    for i in range(BOOTSTRAP_RESAMPLES):
        idx = _rng.integers(0, _n, _n)
        conf = np.zeros((corn.NUM_GRADES, corn.NUM_GRADES))
        np.add.at(conf, (Y_TRUE[idx], Y_PRED[idx]), 1)
        _samples[i] = qwk_from_confusion(conf)
    CI_LOW, CI_HIGH = (float(v) for v in np.percentile(_samples, [2.5, 97.5]))
    BOOTSTRAP = {
        "metric": "quadratic_weighted_kappa",
        "point_estimate": METRICS["qwk"],
        "ci_lower_95": CI_LOW, "ci_upper_95": CI_HIGH,
        "resamples": BOOTSTRAP_RESAMPLES, "seed": BOOTSTRAP_SEED,
        "method": "percentile bootstrap over evaluated images, resampled with replacement",
        "bootstrap_mean": float(_samples.mean()), "bootstrap_std": float(_samples.std()),
        "n_evaluated": int(_n),
    }
    print(f"QWK point estimate : {METRICS['qwk']:.6f}")
    print(f"95% CI             : [{CI_LOW:.4f}, {CI_HIGH:.4f}]")
    print(f"resamples / seed   : {BOOTSTRAP_RESAMPLES} / {BOOTSTRAP_SEED}")
    print(f"bootstrap mean/std : {_samples.mean():.6f} / {_samples.std():.6f}")
    print(f"\nNote: {_n} images is a small sample -- this interval is wide by construction, and "
          "grade 1 has very few cases.")
else:
    BOOTSTRAP = {}
    print("RUN_EXTERNAL_EVALUATION is False -- no interval was computed.")


In [ ]:
# ==== [E12] ARTIFACTS -- per-image table, metrics, provenance (new directory, never overwritten) ====
if RUN_EXTERNAL_EVALUATION:
    EVAL_ROOT = posixpath.join(colab_config.DRIVE.experiments_root, "IDRiD_External_Evaluation")
    RUN_ID = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    OUTPUT_DIR = posixpath.join(EVAL_ROOT, RUN_ID)
    assert_writable(OUTPUT_DIR, "the evaluation output")
    if os.path.exists(OUTPUT_DIR):
        raise RuntimeError(f"{OUTPUT_DIR} already exists -- refusing to overwrite a previous run.")
    os.makedirs(OUTPUT_DIR)          # exist_ok=False: never overwrites
    OUTPUTS = []

    def save_json(name, payload):
        path = posixpath.join(OUTPUT_DIR, name)
        with open(path, "w") as handle:
            json.dump(payload, handle, indent=2,
                      default=lambda v: v.item() if isinstance(v, np.generic)
                      else (v.tolist() if isinstance(v, np.ndarray) else str(v)))
        OUTPUTS.append(path)
        return path

    PER_IMAGE = []
    for position, sample in enumerate(SAMPLES):
        row = {"image_id": sample["image_id"], "source_stem": sample["source_stem"],
               "source_image": sample["source_image"],
               "ground_truth_grade": int(Y_TRUE[position]),
               "predicted_grade": int(Y_PRED[position]),
               "correct": bool(Y_TRUE[position] == Y_PRED[position]),
               "error_distance": int(abs(int(Y_TRUE[position]) - int(Y_PRED[position])))}
        row.update({f"raw_logit_{k}": float(LOGITS[position, k]) for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"cumulative_probability_{k}": float(P_CUM[position, k])
                    for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"class_probability_{g}": float(CLASS_PROBABILITIES[position, g]) for g in GRADES})
        row.update({"RACAF_reliability": float(sample["reliability"]),
                    "evaluation_status": "evaluated", "exclusion_reason": ""})
        PER_IMAGE.append(row)
    for excluded in EXCLUSIONS:
        row = {"image_id": excluded["image_id"], "source_stem": excluded["source_stem"],
               "source_image": posixpath.join(IDRID_TEST_IMAGE_DIR, excluded["source_stem"] + ".jpg"),
               "ground_truth_grade": int(excluded["true_grade"]), "predicted_grade": "",
               "correct": "", "error_distance": ""}
        row.update({f"raw_logit_{k}": "" for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"cumulative_probability_{k}": "" for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"class_probability_{g}": "" for g in GRADES})
        row.update({"RACAF_reliability": "", "evaluation_status": "excluded",
                    "exclusion_reason": excluded["reason"]})
        PER_IMAGE.append(row)

    _csv_path = posixpath.join(OUTPUT_DIR, "per_image_results.csv")
    with open(_csv_path, "w", newline="") as handle:
        writer = _csv.DictWriter(handle, fieldnames=list(PER_IMAGE[0]))
        writer.writeheader()
        writer.writerows(PER_IMAGE)
    OUTPUTS.append(_csv_path)

    _cm_path = posixpath.join(OUTPUT_DIR, "confusion_matrix.csv")
    with open(_cm_path, "w", newline="") as handle:
        writer = _csv.writer(handle)
        writer.writerow(["true_grade"] + [f"predicted_{g}" for g in GRADES])
        for grade in GRADES:
            writer.writerow([grade] + [int(v) for v in CONFUSION[grade]])
    OUTPUTS.append(_cm_path)

    REPORT = {
        "experiment": "IDRiD External Evaluation (Experiment 2)",
        "role": "EXTERNAL EVALUATION of the frozen RACAF model -- no training, no tuning",
        "run_id": RUN_ID, "output_dir": OUTPUT_DIR,
        "environment": ENVIRONMENT,
        "checkpoints": CHECKPOINT_PROVENANCE,
        "dataset": {
            "name": "IDRiD Disease Grading -- Testing Set",
            "image_dir": IDRID_TEST_IMAGE_DIR, "label_csv": IDRID_TEST_LABEL_CSV,
            "label_columns_used": [LABEL_NAME_COLUMN, LABEL_GRADE_COLUMN],
            "label_columns_deliberately_unused": ["Risk of macular edema"],
            "expected_images": EXPECTED_TEST_IMAGES,
            "expected_grade_distribution": EXPECTED_GRADE_DISTRIBUTION,
            "raw_tree_used": True, "processed_tree_used": False,
            "id_namespace": "IDRiDtest_<nnn>",
        },
        "preprocessing": {
            "stage02": "image_preprocessing.preprocess_array(profile='DR'), gamma 1.2, CLAHE clip 2.0 tiles (8,8)",
            "stage03_vessel": "frozen LWNet, FOV crop, 512x512",
            "stage04_lesion": "frozen attention U-Net, (1,512,512,4) = RGB/255 + vessel, 4 lesion channels",
            "stage05_input": [*IMAGE_SIZE, lfed.NUM_CHANNELS],
            "stage06_input": [*jtd.STAGE6_IMAGE_SIZE, 3],
            "augmentation": False, "shuffle": False, "batch_size": BATCH_SIZE,
        },
        "racaf_reliability": RELIABILITY_SUMMARY,
        "corn": {"num_grades": corn.NUM_GRADES, "num_thresholds": corn.NUM_THRESHOLDS,
                 "decode": "count(cumprod(sigmoid(z)) > 0.5)"},
        "metrics": METRICS,
        "calibration": CALIBRATION,
        "bootstrap": BOOTSTRAP,
        "exclusions": EXCLUSIONS,
        "data_integrity": {
            "idrid_train_test_byte_duplicates": [
                {"train": a, "test": b, "sha256": h} for a, b, h in TEST_VS_IDRID_TRAIN],
            "stage04_lesion_training_vs_test_overlap": [
                {"segmentation": a, "test": b, "sha256": h} for a, b, h in SEG_VS_TEST],
            "note": ("IDRiD's own train split is never used by this pipeline, so an internal "
                     "IDRiD duplicate is not leakage for this evaluation. An overlap with the "
                     "Stage 04 lesion-training images WOULD be, and hard-stops the run."),
        },
        "independence_limitation": (
            "IDRiD is external at the CLASSIFICATION level only. The frozen Stage 04 lesion model "
            "was trained on the IDRiD segmentation subset and its loss weights were chosen from "
            "those images, so this is NOT a completely independent full-pipeline evaluation."),
        "checks": {name: {"ok": ok, "detail": detail} for name, (ok, detail) in EVAL_CHECKS.items()},
        "outputs": sorted(OUTPUTS),
    }
    save_json("external_evaluation_report.json", REPORT)
    save_json("metrics.json", METRICS)
    save_json("provenance.json", {"environment": ENVIRONMENT, "checkpoints": CHECKPOINT_PROVENANCE,
                                  "run_id": RUN_ID, "output_dir": OUTPUT_DIR})

    _md = [
        f"# IDRiD External Evaluation -- {RUN_ID}", "",
        f"Frozen RACAF model `{CHECKPOINT_PROVENANCE['racaf_model_sha256'][:16]}...` evaluated on "
        f"{METRICS['evaluated']} IDRiD grading test images ({METRICS['excluded']} excluded).", "",
        "| metric | value |", "|---|---|",
        f"| QWK | {METRICS['qwk']:.6f} |",
        f"| QWK 95% CI | [{BOOTSTRAP['ci_lower_95']:.4f}, {BOOTSTRAP['ci_upper_95']:.4f}] |",
        f"| accuracy | {METRICS['accuracy']:.6f} |",
        f"| balanced accuracy | {METRICS['balanced_accuracy']:.6f} |",
        f"| macro F1 | {METRICS['macro_f1']:.6f} |",
        f"| ordinal MAE | {METRICS['ordinal_mae']:.6f} |",
        f"| Brier | {CALIBRATION['brier_score']:.4f} |",
        f"| ECE (decode) | {CALIBRATION['ece_decode_convention']:.4f} |", "",
        "| grade | support | precision | recall | F1 |", "|---|---|---|---|---|",
    ] + [f"| {g} | {PER_CLASS[g]['support']} | {PER_CLASS[g]['precision']:.4f} | "
         f"{PER_CLASS[g]['recall']:.4f} | {PER_CLASS[g]['f1']:.4f} |" for g in GRADES] + [
        "", "External at the classification level only: the frozen Stage 04 lesion model was "
        "trained and tuned on IDRiD segmentation data. Not an unbiased estimate of "
        "generalization to other cameras, sites or populations. No clinical claim.", ""]
    _md_path = posixpath.join(OUTPUT_DIR, "external_evaluation_report.md")
    with open(_md_path, "w") as handle:
        handle.write("\n".join(_md))
    OUTPUTS.append(_md_path)
    print("\n".join(_md))
    print(f"\nartifacts written to {OUTPUT_DIR}:")
    for path in sorted(set(OUTPUTS)):
        print("   ", posixpath.basename(path))
else:
    OUTPUT_DIR, REPORT, OUTPUTS = None, {}, []
    print("RUN_EXTERNAL_EVALUATION is False -- nothing was written.")


In [ ]:
# ==== [E13] FINAL INTEGRITY CHECK -- the frozen artifacts must be untouched ====
if RUN_EXTERNAL_EVALUATION:
    _after = sha256_file(ARCHIVED_WEIGHTS)
    require("the frozen RACAF weights are unchanged by this evaluation",
            _after == EXPECTED_MODEL_SHA256, _after)
    require("the frozen vessel checkpoint is unchanged",
            sha256_file(VESSEL_CHECKPOINT) == CHECKPOINT_PROVENANCE["vessel_sha256"])
    require("the frozen lesion checkpoint is unchanged",
            sha256_file(LESION_CHECKPOINT) == CHECKPOINT_PROVENANCE["lesion_sha256"])
    require("nothing was written under the frozen experiment or archive",
            not is_protected(OUTPUT_DIR), OUTPUT_DIR)
    require("the APTOS caches were not touched",
            os.path.realpath(IDRID_CACHE_DIR) != os.path.realpath(config.LOCAL_FEATURE_RESULTS_DIR))

    _failed = [name for name, (ok, _) in EVAL_CHECKS.items() if not ok]
    print("\n" + "=" * 78)
    print(f"IDRiD EXTERNAL EVALUATION: {'PASS' if not _failed else 'FAIL'}")
    print(f"  images evaluated : {METRICS['evaluated']}   excluded: {METRICS['excluded']}")
    print(f"  QWK              : {METRICS['qwk']:.6f}  "
          f"95% CI [{BOOTSTRAP['ci_lower_95']:.4f}, {BOOTSTRAP['ci_upper_95']:.4f}]")
    print(f"  accuracy         : {METRICS['accuracy']:.6f}")
    print(f"  model SHA-256    : {CHECKPOINT_PROVENANCE['racaf_model_sha256']}")
    print(f"  output directory : {OUTPUT_DIR}")
    print(f"  checks           : {len(EVAL_CHECKS) - len(_failed)}/{len(EVAL_CHECKS)} passed")
    print("=" * 78)
    if _failed:
        raise RuntimeError(f"FINAL INTEGRITY CHECK FAILED: {_failed}")
    print("The frozen RACAF experiment, its archive and the APTOS caches are untouched.")
    print("Record the numbers above in docs/experiments/IDRiD_External_Evaluation_Report.md.")
else:
    print("RUN_EXTERNAL_EVALUATION is False -- nothing to verify.")
